# The canonical coordinates of the coherent coupler

In this notebook we use our analytic solution to the coherent coupler in the Weierstrass notation to explore coordinate transformations. We show that it is possible to transform the coherent coupler to a canonical coordinate system in which the solutions are single valued functions, namely Kronecker theta functions. In doing so, we show that the classic trick of turning the quartic to a cubic in the elliptic differential equation leaves the differential system of coupled modes invariant in terms of the type of terms that appear but it delicately balances coefficients to kill the quartic term. 

In [110]:
from sympy import *

# -- Symbols --

(x, y, z, t, x0, y0, z0, z1, t0, Z, g2, g3, m, n, l, k, i, j, M, N, C, n2, T) = symbols(
    '''x, y, z, t, x0, y0, z0, z1, t0, Z, g2, g3, m, n, l, k, i, j, M, N, C, n2, T'''
)
(delta, nu, Aeff, chi) = symbols(
    '''delta, nu, Aeff, chi'''
)

gtilde2 = Symbol('gtilde2', latex_name=r'\tilde{g_2}')
gtilde3 = Symbol('gtilde3', latex_name=r'\tilde{g_3}')

# -- Functions --

esp = Function('esp') # Elementary symmetric polynomials

pw = Function('pw') # Weierstrass P function
pwp = Function('pwp') # Derivative of Weierstrass P function
zw = Function('zw') # Weierstrass Zeta function
sigma = Function('sigma') # Weierstrass Sigma function
wp = Function('wp') # Weierstrass P function
wpp = Function('\wp\'') # Derivative of Weierstrass P function
zeta = Function('zeta') # Weierstrass Zeta function

rhop = Function('\\rho\'')
Delta = Function('Delta')
rho = Function('rho')

kappa = Function('kappa')
phi = Function('phi')
varphi = Function('varphi')
h = Function('h')
q = Function('q')
s = Function('s')
u = Function('u')
v = Function('v')
w = Function('w')
ws = Function('ws')
xi = Function('xi')
P = Function('P') # Polynomial
Q = Function('Q') # Polynomial
R = Function('R') # Polynomial

U = Function('U')
V = Function('V')
W = Function('W')
H = Function('H')


uhat = Function('uhat', latex_name=r'\hat{u}')
vhat = Function('vhat', latex_name=r'\hat{v}')
Hhat = Symbol('Hhat', latex_name=r'\hat{H}')

ubar = Function('ubar', latex_name=r'\bar{u}')
vbar = Function('vbar', latex_name=r'\bar{v}')
Hbar = Symbol('Hbar', latex_name=r'\bar{H}')
wbar = Function('wbar', latex_name=r'\bar{w}')
rhobar = Function('rhobar', latex_name=r'\bar{\rho}')

utilde = Function('utilde', latex_name=r'\tilde{u}')
vtilde = Function('vtilde', latex_name=r'\tilde{v}')
Htilde = Symbol('Htilde', latex_name=r'\tilde{H}')
htilde = Function('htilde', latex_name=r'\tilde{h}')
wtilde = Function('wtilde', latex_name=r'\tilde{w}')
rhotilde = Function('rhotilde', latex_name=r'\tilde{\rho}')

f = Function('f')


Summ = Function('Summ')

# -- Indexed Symbols --

Omega = IndexedBase('Omega')
F = IndexedBase('F')
r = IndexedBase('r')
gamma = IndexedBase('gamma')
gammabar = IndexedBase('gammabar', latex_name=r'\bar{\gamma}')
gammatilde = IndexedBase('gammatilde', latex_name=r'\tilde{\gamma}')
epsilontilde = IndexedBase('epsilontilde', latex_name=r'\tilde{\epsilon}')
ebar = IndexedBase('ebar', latex_name=r'\bar{e}')
etilde = IndexedBase('etilde', latex_name=r'\tilde{e}')
mu = IndexedBase('mu')
mubar = IndexedBase('mubar', latex_name=r'\bar{\mu}')
mutilde = IndexedBase('mutilde', latex_name=r'\tilde{\mu}')
nu = IndexedBase('nu')
theta = IndexedBase('theta')
Theta = IndexedBase('Theta')
X = IndexedBase('X')
Y = IndexedBase('Y')
a = IndexedBase('a')
b = IndexedBase('b')
c = IndexedBase('c')
d = IndexedBase('d')
p = IndexedBase('p')
G = IndexedBase('G')
psi = IndexedBase('psi')
Psi = IndexedBase('Psi')
upsilon = IndexedBase('upsilon')
epsilon = IndexedBase('epsilon')
omega = IndexedBase('omega')
alpha = IndexedBase('alpha')
beta = IndexedBase('beta')
K = IndexedBase('K')
lam = IndexedBase('lambda')
Lamda = IndexedBase('Lamda')

wild = Wild('*')


from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

# kth order derivatives of Weierstrass P
from wpk import wpk, wzk, wsk, run_tests

# The package containing mpmath expressions for Weierstrass elliptic functions
from weierstrass_modified import Weierstrass
we = Weierstrass()
from mpmath import exp as mpexp

# Numeric solutions to diff eqs
from numpy import linspace, absolute, angle, square, real, imag, conj, array as arraynp, concatenate
from numpy import vectorize as np_vectorize # not to get confused with vectorise in other packages
import scipy.integrate
import matplotlib.pyplot as plt
from random import random

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Elliptic function identities

In [2]:
sigma_p_identity = Eq(
    wp(y, g2, g3) - wp(x, g2, g3),
    sigma(x + y, g2, g3) * sigma(x - y, g2, g3) / (sigma(x, g2, g3) ** 2 * sigma(y, g2, g3) ** 2) 
)
pw_to_zw_identity = Eq(
    (wpp(z,g2,g3) - wpp(x,g2,g3))/(wp(z,g2,g3) - wp(x,g2,g3))/2,
    zeta(z + x,g2, g3) - zeta(z,g2, g3) - zeta(x,g2, g3)
)
pw_to_zw_identity_one_sided = Eq(
    wpp(z, g2, g3)/(wp(x, g2, g3) - wp(z, g2, g3)),
    2*zeta(z, g2, g3) - zeta(-x + z, g2, g3) - zeta(x + z, g2, g3)
)

zw_dlog_sigma_identity = Eq(zeta(z - x, g2, g3), Derivative(log(sigma(z - x, g2, g3)), z))
pw_to_dlog_sigma_identity = pw_to_zw_identity.subs([
    zw_dlog_sigma_identity.subs(x,-x).args,
    zw_dlog_sigma_identity.subs(x,0).args,
])
pw_to_dlog_sigma_identity_b = pw_to_dlog_sigma_identity.subs(x,-x).subs([
    (wp(-x,g2,g3), wp(x,g2,g3)), (wpp(-x,g2,g3), -wpp(x,g2,g3)), (zeta(-x, g2,g3), -zeta(x, g2,g3))
])

pw_addition_id = Eq(
    wp(x + y, g2, g3),
    -wp(x, g2, g3) - wp(y, g2, g3) + (wpp(x, g2, g3) - wpp(y, g2, g3))**2/(4*(wp(x, g2, g3) - wp(y, g2, g3))**2)
)

pwp_sigma_dbl_ratio = Eq(wpp(z,g2,g3), - sigma(2*z,g2,g3)/sigma(z,g2,g3)**4)



# See Homogenity 
# https://functions.wolfram.com/EllipticFunctions/WeierstrassP/introductions/Weierstrass/ShowAll.html
sig_scale_law = Eq(sigma(x,g2*c**4,g3*c**6), sigma(x*c,g2,g3)/c)
zw_scale_law = Eq(zeta(x,g2*c**4,g3*c**6), zeta(x*c,g2,g3)*c)
pw_scale_law = Eq(wp(x,g2*c**4,g3*c**6), wp(x*c,g2,g3)*c**2)
pwp_scale_law = Eq(wpp(x,g2*c**4,g3*c**6), wpp(x*c,g2,g3)*c**3)

omega1_scale_law_a = Eq(omega[1], f(1, g2,g3))
omega1_scale_law_b = Eq(c*omega[1], f(1, g2/c**4,g3/c**6))
omega3_scale_law_a = Eq(omega[3], f(3, g2,g3))
omega3_scale_law_b = Eq(c*omega[3], f(3, g2/c**4,g3/c**6))

sigma_p_identity
pw_to_zw_identity
pw_to_zw_identity_one_sided
zw_dlog_sigma_identity
pw_to_dlog_sigma_identity_b
pw_addition_id
pwp_sigma_dbl_ratio

sig_scale_law
zw_scale_law
pw_scale_law
pwp_scale_law
omega1_scale_law_a
omega1_scale_law_b
omega3_scale_law_a
omega3_scale_law_b

Eq(-wp(x, g2, g3) + wp(y, g2, g3), sigma(x - y, g2, g3)*sigma(x + y, g2, g3)/(sigma(x, g2, g3)**2*sigma(y, g2, g3)**2))

Eq((-\wp'(x, g2, g3) + \wp'(z, g2, g3))/(2*(-wp(x, g2, g3) + wp(z, g2, g3))), -zeta(x, g2, g3) - zeta(z, g2, g3) + zeta(x + z, g2, g3))

Eq(\wp'(z, g2, g3)/(wp(x, g2, g3) - wp(z, g2, g3)), 2*zeta(z, g2, g3) - zeta(-x + z, g2, g3) - zeta(x + z, g2, g3))

Eq(zeta(-x + z, g2, g3), Derivative(log(sigma(-x + z, g2, g3)), z))

Eq((\wp'(x, g2, g3) + \wp'(z, g2, g3))/(2*(-wp(x, g2, g3) + wp(z, g2, g3))), zeta(x, g2, g3) - Derivative(log(sigma(z, g2, g3)), z) + Derivative(log(sigma(-x + z, g2, g3)), z))

...

Eq(\wp'(x, g2*c**4, g3*c**6), \wp'(x*c, g2, g3)*c**3)

Eq(omega[1], f(1, g2, g3))

Eq(omega[1]*c, f(1, g2/c**4, g3/c**6))

Eq(omega[3], f(3, g2, g3))

Eq(omega[3]*c, f(3, g2/c**4, g3/c**6))

## The solution in original coordinates

We start by reminding ourselves of the solutions derived in *The general 2 mode case of the coherent coupler* notebook for modes in the original coordinates $u,v$:

In [3]:
Lams = [
    Eq(
        Lamda[0, m],
        -a[m] - gamma[m]*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2))
        + Sum(a[j, j]*gamma[j], (j, 1, 2))/2 - Sum(a[m, k]*gamma[k], (k, 1, 2)) + Sum(a[j]/2, (j, 1, 2))
    ),
    Eq(Lamda[1, m], Sum(a[m, k], (k, 1, 2)) - Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)))
]

u_sqrt_wp_z0_z1 = Eq(
    u(z, mu[j]),
    d[4]**Rational(-1, 4)*
    sqrt(wpp(z1, g2, g3))/
    sqrt((wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3)))/
    sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))
    *sigma(z - 2*z0 + mu[j], g2, g3)/(sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3))
    *exp(z*r[0, j] + log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] + epsilon[j])
)
v_sqrt_wp_z0_z1 = Eq(
    v(z, mu[j]),
    d[4]**Rational(-1, 4)*
    sqrt(wpp(z1, g2, g3))/
    sqrt((wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3)))/
    sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))
    *sigma(z - mu[j], g2, g3)/(sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3))
    *exp(-z*r[0, j] - log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] - epsilon[j])
)

sum_r1j_1 = Eq(Sum(r[1, j], (j, 1, 2)), 1)
r1j_log_part = Eq(r[1, j], Lamda[1, j]/sqrt(d[4]))

u_sqrt_wp_z0_z1
v_sqrt_wp_z0_z1
sum_r1j_1
r1j_log_part

for _ in Lams:
    _

Eq(u(z, mu[j]), sqrt(\wp'(z1, g2, g3))*sigma(z - 2*z0 + mu[j], g2, g3)*exp(z*r[0, j] + log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] + epsilon[j])/(sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)*d[4]**(1/4)))

Eq(v(z, mu[j]), sqrt(\wp'(z1, g2, g3))*sigma(z - mu[j], g2, g3)*exp(-z*r[0, j] - log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] - epsilon[j])/(sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)*d[4]**(1/4)))

Eq(Sum(r[1, j], (j, 1, 2)), 1)

Eq(r[1, j], Lamda[1, j]/sqrt(d[4]))

Eq(Lamda[0, m], -a[m] - gamma[m]*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)) + Sum(a[j, j]*gamma[j], (j, 1, 2))/2 - Sum(a[m, k]*gamma[k], (k, 1, 2)) + Sum(a[j]/2, (j, 1, 2)))

Eq(Lamda[1, m], Sum(a[m, k], (k, 1, 2)) - Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)))

### The original parameters

In [4]:
b_j_coeffs = [
    Eq(
        b[0], 
        (gamma[1] - gamma[2])**2*(Sum(a[j, j]/4, (j, 1, 2)) - Sum(a[j, k]/8, (j, 1, 2), (k, 1, 2))) 
        + a[0] + Sum(a[j]*gamma[j], (j, 1, 2))
    ),
    Eq(b[1], -Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j], (j, 1, 2))),
    Eq(b[2], Sum(a[j, k]/2, (j, 1, 2), (k, 1, 2)))
]

c_j_coeffs = [Eq(c[0], Product(gamma[j], (j, 1, 2))), Eq(c[1], 0), Eq(c[2], 1)]


d_j_coeffs = [
    Eq(d[0], b[0]**2 - 4*c[0]),
    Eq(d[1], 2*b[0]*b[1] - 4*c[1]),
    Eq(d[2], 2*b[0]*b[2] + b[1]**2 - 4*c[2]),
    Eq(d[3], 2*b[1]*b[2]),
    Eq(d[4], b[2]**2)
]

g2_dj = Eq(g2, d[0]*d[4] - d[1]*d[3]/4 + d[2]**2/12)
g3_dj = Eq(g3, d[0]*d[2]*d[4]/6 - d[0]*d[3]**2/16 - d[1]**2*d[4]/16 + d[1]*d[2]*d[3]/48 - d[2]**3/216)

for _ in b_j_coeffs:
    _
    
for _ in c_j_coeffs:
    _
    
for _ in d_j_coeffs:
    _

g2_dj
g3_dj

Eq(b[0], (gamma[1] - gamma[2])**2*(Sum(a[j, j]/4, (j, 1, 2)) - Sum(a[j, k]/8, (j, 1, 2), (k, 1, 2))) + a[0] + Sum(a[j]*gamma[j], (j, 1, 2)))

Eq(b[1], -Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j], (j, 1, 2)))

Eq(b[2], Sum(a[j, k]/2, (j, 1, 2), (k, 1, 2)))

Eq(c[0], Product(gamma[j], (j, 1, 2)))

Eq(c[1], 0)

...

Eq(d[2], 2*b[0]*b[2] + b[1]**2 - 4*c[2])

Eq(d[3], 2*b[1]*b[2])

Eq(d[4], b[2]**2)

Eq(g2, d[0]*d[4] - d[1]*d[3]/4 + d[2]**2/12)

Eq(g3, d[0]*d[2]*d[4]/6 - d[0]*d[3]**2/16 - d[1]**2*d[4]/16 + d[1]*d[2]*d[3]/48 - d[2]**3/216)

We recall the equations from the original derivation of the solution in terms of $\rho, b_j, \gamma_j, \lambda_1$:

In [5]:
drhop_b = Eq(
    Derivative(rho(z), z)**2, 
    (rho(z)**2*b[2] + rho(z)*b[1] + b[0])**2 - 4*Product(-rho(z) + gamma[j], (j, 1, 2))
)
drhop_d = Eq(Derivative(rho(z), z)**2, Sum(rho(z)**j*d[j], (j, 0, 4)))

drho_2z = Eq(Derivative(rho(z), (z, 2)), Sum(j*rho(z)**(j-1)*d[j]/2, (j, 0, 4)))
drho_2z_b = Eq(
    Derivative(rho(z), (z, 2)),
    (2*rho(z)*b[2] + b[1])*(rho(z)**2*b[2] + rho(z)*b[1] + b[0]) + 2*(-2*rho(z) + gamma[1] + gamma[2])
)

drhop_b
drhop_d
drho_2z
drho_2z_b

Eq(Derivative(rho(z), z)**2, (rho(z)**2*b[2] + rho(z)*b[1] + b[0])**2 - 4*Product(-rho(z) + gamma[j], (j, 1, 2)))

Eq(Derivative(rho(z), z)**2, Sum(rho(z)**j*d[j], (j, 0, 4)))

Eq(Derivative(rho(z), (z, 2)), Sum(j*rho(z)**(j - 1)*d[j]/2, (j, 0, 4)))

Eq(Derivative(rho(z), (z, 2)), (2*rho(z)*b[2] + b[1])*(rho(z)**2*b[2] + rho(z)*b[1] + b[0]) - 4*rho(z) + 2*gamma[1] + 2*gamma[2])

### Reorganising the original equations of motion

In [6]:
ajk_syms = [(a[2, 1], a[1, 2])]

uv_j_rho = Eq(u(z,mu[j])*v(z,mu[j]), gamma[j] - rho(z))

duvj = Eq(
    Derivative(u(z, mu[j])*v(z, mu[j]),z), 
    Product(v(z, mu[k]), (k, 1, 2)) - Product(u(z, mu[k]), (k, 1, 2))
)

uprodj = Eq(
    Product(u(z, mu[j]), (j, 1, 2)),
    -Derivative(u(z, mu[j])*v(z, mu[j]), z)/2 + a[0]/2 + 
    Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2))/2 + 
    Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2))/2
)
vprodj = Eq(uprodj.lhs + duvj.rhs.replace(k,j), uprodj.rhs + duvj.lhs.replace(k,j))
a_0_eq = Eq(uprodj.rhs + vprodj.rhs, uprodj.lhs + vprodj.lhs)

du_phase_mod_part = (a[j] + Sum(a[j,k]*u(z,mu[k])*v(z,mu[k]), (k,1,2)))*u(z,mu[j])
dv_phase_mod_part = -(a[j] + Sum(a[j,k]*u(z,mu[k])*v(z,mu[k]), (k,1,2)))*v(z,mu[j])

du_mixing_part = Product(v(z,mu[k]), (k,1,2))/v(z, mu[j])
dv_mixing_part = -Product(u(z,mu[k]), (k,1,2))/u(z, mu[j])

duj = Eq(diff(u(z,mu[j]),z), -du_phase_mod_part + du_mixing_part)
dvj = Eq(diff(v(z,mu[j]),z), (-dv_phase_mod_part).factor() + dv_mixing_part)
duj_subs = [duj.subs(j, _j + 1).args for _j in range(2)]
dvj_subs = [dvj.subs(j, _j + 1).args for _j in range(2)]
duj_dvj_subs = duj_subs + dvj_subs


assert (
    diff(solve(a_0_eq.doit(),a[0])[0],z).subs(duj_dvj_subs).doit().subs(ajk_syms).simplify() == 0
), 'a0 not conserved'

uv_j_rho

uprodj
vprodj
a_0_eq

duj
dvj
duvj

Eq(u(z, mu[j])*v(z, mu[j]), -rho(z) + gamma[j])

Eq(Product(u(z, mu[j]), (j, 1, 2)), -Derivative(u(z, mu[j])*v(z, mu[j]), z)/2 + a[0]/2 + Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2))/2 + Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2))/2)

Eq(Product(v(z, mu[j]), (j, 1, 2)), Derivative(u(z, mu[j])*v(z, mu[j]), z)/2 + a[0]/2 + Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2))/2 + Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2))/2)

Eq(a[0] + Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2)) + Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2)), Product(u(z, mu[j]), (j, 1, 2)) + Product(v(z, mu[j]), (j, 1, 2)))

Eq(Derivative(u(z, mu[j]), z), -(a[j] + Sum(u(z, mu[k])*v(z, mu[k])*a[j, k], (k, 1, 2)))*u(z, mu[j]) + Product(v(z, mu[k]), (k, 1, 2))/v(z, mu[j]))

Eq(Derivative(v(z, mu[j]), z), (a[j] + Sum(u(z, mu[k])*v(z, mu[k])*a[j, k], (k, 1, 2)))*v(z, mu[j]) - Product(u(z, mu[k]), (k, 1, 2))/u(z, mu[j]))

Eq(Derivative(u(z, mu[j])*v(z, mu[j]), z), -Product(u(z, mu[k]), (k, 1, 2)) + Product(v(z, mu[k]), (k, 1, 2)))

## Gauge transform to the hat coordinates

In this section we implement a gauge transform to remove the log of $\sigma$ terms from the exponentials in the solutions, i.e., we wish to remove the essential singularities to leave meromorphic functions. This worked well in the FWM case and we want to see what the analagous process is for the coherent coupler.

In [111]:
uU_sub = Eq(
    u(z,mu[j]), 
    uhat(z, mu[j])
    *exp(-varphi(z,j))
)
vV_sub = Eq(
    v(z,mu[j]), 
    vhat(z, mu[j])
    *exp(varphi(z,j))
)

Uu_sub = Eq(uU_sub.rhs*exp(varphi(z, j)), uU_sub.lhs*exp(varphi(z, j)))
Vv_sub = Eq(vV_sub.rhs*exp(-varphi(z, j)), vV_sub.lhs*exp(-varphi(z, j)))

uv_gamma_cons =Eq(
    uv_j_rho.lhs - uv_j_rho.lhs.subs(j,k),
    uv_j_rho.rhs - uv_j_rho.rhs.subs(j,k)
)

uv_gamma_cons_gauge = Eq(
    uv_gamma_cons.lhs.subs([
        uU_sub.args, 
        vV_sub.args,
        uU_sub.subs(j,k).args, 
        vV_sub.subs(j,k).args
    ]).simplify(),
    uv_gamma_cons.rhs
)
uv_gam_gauge_k_to_j = Eq(
    uhat(z, mu[j])*vhat(z, mu[j]) - uv_gamma_cons_gauge.lhs,
    uhat(z, mu[j])*vhat(z, mu[j]) - uv_gamma_cons_gauge.rhs
)


uU_sub
vV_sub

Uu_sub
Vv_sub

uv_j_rho
uv_gamma_cons
uv_gamma_cons_gauge

Eq(u(z, mu[j]), uhat(z, mu[j])*exp(-varphi(z, j)))

Eq(v(z, mu[j]), vhat(z, mu[j])*exp(varphi(z, j)))

Eq(uhat(z, mu[j]), u(z, mu[j])*exp(varphi(z, j)))

Eq(vhat(z, mu[j]), v(z, mu[j])*exp(-varphi(z, j)))

Eq(u(z, mu[j])*v(z, mu[j]), -rho(z) + gamma[j])

Eq(u(z, mu[j])*v(z, mu[j]) - u(z, mu[k])*v(z, mu[k]), gamma[j] - gamma[k])

Eq(uhat(z, mu[j])*vhat(z, mu[j]) - uhat(z, mu[k])*vhat(z, mu[k]), gamma[j] - gamma[k])

### Exploring the required phase shift

Before we define the $\varphi$ shift in terms of integrals of modes $u,v$, we want to see what the required shift is in terms of an integral over $\rho$. In general we require two things from the phase shifts applied to the modes:

- they cancel the essential singularities from our analytic solutions
- they sum to zero so that our differential equations are still polynomial like and we do not introduce new complicated integral terms in exponents

It is not too difficult to guess a form to meet these requirement. Something like the below appears to work, with $\beta_0$ some yet to be fixed constant:

In [116]:
phi_opt_1 = beta[0]*z + log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*(-r[1, 1]+r[1,2])/2

phi_j_eqs = [
    Eq(varphi(z,1), phi_opt_1),
    Eq(varphi(z,2), -phi_opt_1)
]
phi_j_subs = [_.args for _ in phi_j_eqs]

gauge_phi_sum = Eq(
    Sum(varphi(z,j),(j,1,2)), 
    Sum(varphi(z,j),(j,1,2)).doit().subs(phi_j_subs)
)

sum_r1j_1
for _ in phi_j_eqs:
    _

gauge_phi_sum

Eq(Sum(r[1, j], (j, 1, 2)), 1)

Eq(varphi(z, 1), z*beta[0] + (-r[1, 1] + r[1, 2])*log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/2)

Eq(varphi(z, 2), -z*beta[0] - (-r[1, 1] + r[1, 2])*log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/2)

Eq(Sum(varphi(z, j), (j, 1, 2)), 0)

In [202]:
uhat_gauge_1 = Eq(
    Uu_sub.lhs.subs(j,1),
    Uu_sub.rhs.subs([
        u_sqrt_wp_z0_z1.args,
        (j,1),
        phi_j_subs[0]
    ]).expand().simplify().subs([
        (r[1,2], solve(sum_r1j_1.doit(), r[1,2])[0]),
        sigma_p_identity.subs([(y,z1), (x, z-z0)]).args
    ]).simplify()
)

vhat_gauge_1 = Eq(
    Vv_sub.lhs.subs(j,1),
    Vv_sub.rhs.subs([
        v_sqrt_wp_z0_z1.args,
        (j,1),
        phi_j_subs[0]
    ]).expand().simplify().subs([
        (r[1,2], solve(sum_r1j_1.doit(), r[1,2])[0]),
        sigma_p_identity.subs([(y,z1), (x, z-z0)]).args
    ]).simplify()
)

uhat_gauge_2 = Eq(
    Uu_sub.lhs.subs(j,2),
    Uu_sub.rhs.subs([
        u_sqrt_wp_z0_z1.args,
        (j,2),
        phi_j_subs[1]
    ]).expand().simplify().subs([
        (r[1,2], solve(sum_r1j_1.doit(), r[1,2])[0]),
        sigma_p_identity.subs([(y,z1), (x, z-z0)]).args
    ]).simplify()
)

vhat_gauge_2 = Eq(
    Vv_sub.lhs.subs(j,2),
    Vv_sub.rhs.subs([
        v_sqrt_wp_z0_z1.args,
        (j,2),
        phi_j_subs[1]
    ]).expand().simplify().subs([
        (r[1,2], solve(sum_r1j_1.doit(), r[1,2])[0]),
        sigma_p_identity.subs([(y,z1), (x, z-z0)]).args
    ]).simplify()
)

# upsilon captures the sign ambiguity

sig_sqrt_1 = Eq(
    sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/
    sqrt(
        sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/
        (sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2)
    ),
    upsilon*sigma(z1, g2, g3)*sigma(z - z0, g2, g3)/sigma(z - z0 - z1, g2, g3)
)

sig_sqrt_2 = Eq(
    1/(
        sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*
        sqrt(
            sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/
            (sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2)
        )
    ),
    sigma(z1, g2, g3)*sigma(z - z0, g2, g3)/sigma(z - z0 + z1, g2, g3)/upsilon
)
assert (sig_sqrt_1.lhs*sig_sqrt_2.lhs/(sig_sqrt_1.rhs*sig_sqrt_2.rhs)) == 1, 'sqrt ratios not 1'
sig_sqrt_subs = [
    sig_sqrt_1.args,
    sig_sqrt_2.args
]


uhat_gauge_1_b = uhat_gauge_1.subs(sig_sqrt_subs)
vhat_gauge_1_b = vhat_gauge_1.subs(sig_sqrt_subs)
uhat_gauge_2_b = uhat_gauge_2.subs(sig_sqrt_subs)
vhat_gauge_2_b = vhat_gauge_2.subs(sig_sqrt_subs)

uv_hat_gauge_bs = [
    uhat_gauge_1_b,
    vhat_gauge_1_b,
    uhat_gauge_2_b,
    vhat_gauge_2_b
]


uhat_gauge_1
vhat_gauge_1
uhat_gauge_2
vhat_gauge_2

sig_sqrt_1
sig_sqrt_2

for _ in uv_hat_gauge_bs:
    _

Eq(uhat(z, mu[1]), sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*sqrt(\wp'(z1, g2, g3))*sigma(z - 2*z0 + mu[1], g2, g3)*exp(z*beta[0] + z*r[0, 1] + epsilon[1])/(sqrt(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/(sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[1], g2, g3)*d[4]**(1/4)))

Eq(vhat(z, mu[1]), sqrt(\wp'(z1, g2, g3))*sigma(z - mu[1], g2, g3)*exp(-z*beta[0] - z*r[0, 1] - epsilon[1])/(sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*sqrt(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/(sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[1], g2, g3)*d[4]**(1/4)))

Eq(uhat(z, mu[2]), sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*sqrt(\wp'(z1, g2, g3))*sigma(z - 2*z0 + mu[2], g2, g3)*exp(-z*beta[0] + z*r[0, 2] + epsilon[2])/(sqrt(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/(sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[2], g2, g3)*d[4]**(1/4)))

Eq(vhat(z, mu[2]), sqrt(\wp'(z1, g2, g3))*sigma(z - mu[2], g2, g3)*exp(z*beta[0] - z*r[0, 2] - epsilon[2])/(sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*sqrt(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/(sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[2], g2, g3)*d[4]**(1/4)))

Eq(sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/sqrt(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/(sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2)), sigma(z1, g2, g3)*sigma(z - z0, g2, g3)*upsilon/sigma(z - z0 - z1, g2, g3))

Eq(1/(sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*sqrt(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/(sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2))), sigma(z1, g2, g3)*sigma(z - z0, g2, g3)/(sigma(z - z0 + z1, g2, g3)*upsilon))

Eq(uhat(z, mu[1]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - 2*z0 + mu[1], g2, g3)*exp(z*beta[0] + z*r[0, 1] + epsilon[1])*upsilon/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(-z0 + mu[1], g2, g3)*sigma(z - z0 - z1, g2, g3)*d[4]**(1/4)))

Eq(vhat(z, mu[1]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - mu[1], g2, g3)*exp(-z*beta[0] - z*r[0, 1] - epsilon[1])/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(-z0 + mu[1], g2, g3)*sigma(z - z0 + z1, g2, g3)*d[4]**(1/4)*upsilon))

Eq(uhat(z, mu[2]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - 2*z0 + mu[2], g2, g3)*exp(-z*beta[0] + z*r[0, 2] + epsilon[2])*upsilon/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(-z0 + mu[2], g2, g3)*sigma(z - z0 - z1, g2, g3)*d[4]**(1/4)))

Eq(vhat(z, mu[2]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - mu[2], g2, g3)*exp(z*beta[0] - z*r[0, 2] - epsilon[2])/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(-z0 + mu[2], g2, g3)*sigma(z - z0 + z1, g2, g3)*d[4]**(1/4)*upsilon))

In terms of $a_{j,k}$ such a shift would be:

In [118]:
r1j_a_d4 = r1j_log_part.subs(*Lams[1].replace(j,l).replace(m,j).args)

r11_min_r22 = Eq(
    r1j_a_d4.lhs.subs(j,1).doit() - r1j_a_d4.lhs.subs(j,2).doit(),
    (r1j_a_d4.rhs.subs(j,1).doit() - r1j_a_d4.rhs.subs(j,2).doit())
    .simplify().subs(ajk_syms)
)

r1j_a_d4
r11_min_r22

phi_j_eqs[0]
phi_j_eqs[0].subs([r11_min_r22.args])

Eq(r[1, j], (Sum(a[j, k], (k, 1, 2)) - Sum(a[l, k]/4, (l, 1, 2), (k, 1, 2)))/sqrt(d[4]))

Eq(r[1, 1] - r[1, 2], (a[1, 1] - a[2, 2])/sqrt(d[4]))

Eq(varphi(z, 1), z*beta[0] + (-r[1, 1] + r[1, 2])*log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/2)

Eq(varphi(z, 1), z*beta[0] - (a[1, 1] - a[2, 2])*log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/(2*sqrt(d[4])))

and we can get such log terms from modal power integrals such as:

In [132]:
rho_integral = Eq(
    Integral(-rho(z) + rho(mu[j]),z),
    z*(zeta(-z0 + z1 + mu[j], g2, g3) + zeta(z0 + z1 - mu[j], g2, g3))/sqrt(d[4]) -
    log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/sqrt(d[4]) + C
)

rho_integral

Eq(Integral(-rho(z) + rho(mu[j]), z), C + z*(zeta(-z0 + z1 + mu[j], g2, g3) + zeta(z0 + z1 - mu[j], g2, g3))/sqrt(d[4]) - log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/sqrt(d[4]))

## In terms of $u, v$

In this section we now consider a $\varphi$ shift in terms of $u,v$. Inspired by the FWM case, we try the shift in which the modal power weight is the distance from the mean of that matrix vector and similarly in the linear $z$ term. We include $\epsilon_0$ as a constant yet to be fixed:

In [154]:
gauge_phi = Eq(
    varphi(z, j), 
    (a[j] - Sum(a[l], (l, 1, 2))/2 - epsilon[0]*0)*z +
    Sum((a[j, k] - Sum(a[l, k]/2, (l, 1, 2)))*Integral(uhat(z, mu[k])*vhat(z, mu[k]),z), (k, 1, 2))
)
gauge_phi

Eq(varphi(z, j), z*(a[j] - Sum(a[l], (l, 1, 2))/2) + Sum((a[j, k] - Sum(a[l, k]/2, (l, 1, 2)))*Integral(uhat(z, mu[k])*vhat(z, mu[k]), z), (k, 1, 2)))

We confirm that this choice for $\phi$ can deliver the required log term to cancel essential singularities:

In [155]:
_int_subs = [
    (
        Integral(uhat(z, mu[_j+1])*vhat(z, mu[_j+1]),z), 
        Integral(rho(mu[_j+1]) - rho(z),z).subs([rho_integral.subs(j,_j+1).args])
    )
    for _j in range(2)
]

gauge_phi_logs = [
    Eq(
        gauge_phi.lhs.subs(j, _j + 1),
        gauge_phi.rhs.subs(j, _j + 1)
        .doit().subs(_int_subs)
        .subs(ajk_syms)
        .expand()
        .collect([log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3)), sqrt(d[4])], factor)
    )
    for _j in range(2)
]

assert (
    gauge_phi_logs[0].rhs.coeff(log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))) -
    phi_j_eqs[0].subs([r11_min_r22.args]).rhs.coeff(log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3)))
) == 0, 'log coeffs do not match for j=1'
assert (
    gauge_phi_logs[1].rhs.coeff(log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))) -
    phi_j_eqs[1].subs([r11_min_r22.args]).rhs.coeff(log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3)))
) == 0, 'log coeffs do not match for j=2'

for _ in gauge_phi_logs:
    _

Eq(varphi(z, 1), z*(zeta(-z0 + z1 + mu[1], g2, g3)*a[1, 1] - zeta(-z0 + z1 + mu[1], g2, g3)*a[1, 2] + zeta(-z0 + z1 + mu[2], g2, g3)*a[1, 2] - zeta(-z0 + z1 + mu[2], g2, g3)*a[2, 2] + zeta(z0 + z1 - mu[1], g2, g3)*a[1, 1] - zeta(z0 + z1 - mu[1], g2, g3)*a[1, 2] + zeta(z0 + z1 - mu[2], g2, g3)*a[1, 2] - zeta(z0 + z1 - mu[2], g2, g3)*a[2, 2])/(2*sqrt(d[4])) - (a[1, 1] - a[2, 2])*log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/(2*sqrt(d[4])) + (C*a[1, 1] - C*a[2, 2] + z*a[1] - z*a[2])/2)

Eq(varphi(z, 2), -z*(zeta(-z0 + z1 + mu[1], g2, g3)*a[1, 1] - zeta(-z0 + z1 + mu[1], g2, g3)*a[1, 2] + zeta(-z0 + z1 + mu[2], g2, g3)*a[1, 2] - zeta(-z0 + z1 + mu[2], g2, g3)*a[2, 2] + zeta(z0 + z1 - mu[1], g2, g3)*a[1, 1] - zeta(z0 + z1 - mu[1], g2, g3)*a[1, 2] + zeta(z0 + z1 - mu[2], g2, g3)*a[1, 2] - zeta(z0 + z1 - mu[2], g2, g3)*a[2, 2])/(2*sqrt(d[4])) + (a[1, 1] - a[2, 2])*log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/(2*sqrt(d[4])) - (C*a[1, 1] - C*a[2, 2] + z*a[1] - z*a[2])/2)

### Effect on the differential equations

In this section we explore the new differential equations which, by design, we know must have meromorphic solutions:

In [157]:
gauge_subs_phi1 = [
    uU_sub.subs(j,1).args,
    vV_sub.subs(j,1).args,
    uU_sub.subs(j,2).args,
    vV_sub.subs(j,2).args,
]
gauge_subs_phi1_vals = [
    gauge_phi.subs(j,1).doit().args,
    gauge_phi.subs(j,2).doit().args
]


duhat_1_phi = Eq(
    diff(uhat(z, mu[j]), z).subs(j, 1), 
    (
        solve(duj.subs(j, 1).doit().subs(gauge_subs_phi1).doit(), diff(uhat(z, mu[j]), z).subs(j, 1))[0]
        .subs(gauge_subs_phi1_vals).doit().simplify()
        - vhat(z, mu[2])
    ).factor()
    + vhat(z, mu[2])
    
)
duhat_2_phi = Eq(
    diff(uhat(z, mu[j]), z).subs(j, 2), 
    (
        solve(duj.subs(j, 2).doit().subs(gauge_subs_phi1).doit(), diff(uhat(z, mu[j]), z).subs(j, 2))[0]
        .subs(gauge_subs_phi1_vals).doit().simplify()
        - vhat(z, mu[1])
    ).factor()
    + vhat(z, mu[1])
    
)
dvhat_1_phi = Eq(
    diff(vhat(z, mu[j]), z).subs(j, 1), 
    (
        solve(dvj.subs(j, 1).doit().subs(gauge_subs_phi1).doit(), diff(vhat(z, mu[j]), z).subs(j, 1))[0]
        .subs(gauge_subs_phi1_vals).doit().simplify()
        + uhat(z, mu[2])
    ).factor()
    - uhat(z, mu[2])
    
)
dvhat_2_phi = Eq(
    diff(vhat(z, mu[j]), z).subs(j, 2), 
    (
        solve(dvj.subs(j, 2).doit().subs(gauge_subs_phi1).doit(), diff(vhat(z, mu[j]), z).subs(j, 2))[0]
        .subs(gauge_subs_phi1_vals).doit().simplify()
        + uhat(z, mu[1])
    ).factor()
    - uhat(z, mu[1])
    
)
du_dv_hat_eqs = [
    duhat_1_phi,
    duhat_2_phi,
    dvhat_1_phi,
    dvhat_2_phi
]

for _ in du_dv_hat_eqs:
    _


Eq(Derivative(uhat(z, mu[1]), z), -(uhat(z, mu[1])*vhat(z, mu[1])*a[1, 1] + uhat(z, mu[1])*vhat(z, mu[1])*a[2, 1] + uhat(z, mu[2])*vhat(z, mu[2])*a[1, 2] + uhat(z, mu[2])*vhat(z, mu[2])*a[2, 2] + a[1] + a[2])*uhat(z, mu[1])/2 + vhat(z, mu[2]))

Eq(Derivative(uhat(z, mu[2]), z), -(uhat(z, mu[1])*vhat(z, mu[1])*a[1, 1] + uhat(z, mu[1])*vhat(z, mu[1])*a[2, 1] + uhat(z, mu[2])*vhat(z, mu[2])*a[1, 2] + uhat(z, mu[2])*vhat(z, mu[2])*a[2, 2] + a[1] + a[2])*uhat(z, mu[2])/2 + vhat(z, mu[1]))

Eq(Derivative(vhat(z, mu[1]), z), (uhat(z, mu[1])*vhat(z, mu[1])*a[1, 1] + uhat(z, mu[1])*vhat(z, mu[1])*a[2, 1] + uhat(z, mu[2])*vhat(z, mu[2])*a[1, 2] + uhat(z, mu[2])*vhat(z, mu[2])*a[2, 2] + a[1] + a[2])*vhat(z, mu[1])/2 - uhat(z, mu[2]))

Eq(Derivative(vhat(z, mu[2]), z), (uhat(z, mu[1])*vhat(z, mu[1])*a[1, 1] + uhat(z, mu[1])*vhat(z, mu[1])*a[2, 1] + uhat(z, mu[2])*vhat(z, mu[2])*a[1, 2] + uhat(z, mu[2])*vhat(z, mu[2])*a[2, 2] + a[1] + a[2])*vhat(z, mu[2])/2 - uhat(z, mu[1]))

In [194]:
uv_phase_term_rho = Eq(
    -2*du_dv_hat_eqs[0].rhs.coeff(uhat(z, mu[1])),
    -2*du_dv_hat_eqs[0].rhs.coeff(uhat(z, mu[1]))
    .subs([
        (a[2], solve(b_j_coeffs[1].doit(), a[2])[0]),
        (uhat(z, mu[1])*vhat(z, mu[1]), gamma[1] - rho(z)),
        (uhat(z, mu[2])*vhat(z, mu[2]), gamma[2] - rho(z)),
        (gamma[2], - gamma[1])
    ])
    .subs(ajk_syms)
    .expand().collect(rho(z), factor)
    .subs(
        b_j_coeffs[2].rhs.doit().subs(ajk_syms).factor(), 
        b_j_coeffs[2].lhs
    )
)

uv_phase_term_uv = Eq(
    -2*uv_phase_term_rho.lhs, 
    -2*uv_phase_term_rho.rhs.subs([
        (rho(z), -(uhat(z, mu[1])*vhat(z, mu[1]) + uhat(z, mu[2])*vhat(z, mu[2]))/2)
    ])
    .collect(b[2], factor)
)


uv_phase_term_rho
uv_phase_term_uv

Eq(uhat(z, mu[1])*vhat(z, mu[1])*a[1, 1] + uhat(z, mu[1])*vhat(z, mu[1])*a[2, 1] + uhat(z, mu[2])*vhat(z, mu[2])*a[1, 2] + uhat(z, mu[2])*vhat(z, mu[2])*a[2, 2] + a[1] + a[2], -2*rho(z)*b[2] - b[1])

Eq(-2*uhat(z, mu[1])*vhat(z, mu[1])*a[1, 1] - 2*uhat(z, mu[1])*vhat(z, mu[1])*a[2, 1] - 2*uhat(z, mu[2])*vhat(z, mu[2])*a[1, 2] - 2*uhat(z, mu[2])*vhat(z, mu[2])*a[2, 2] - 2*a[1] - 2*a[2], -2*(uhat(z, mu[1])*vhat(z, mu[1]) + uhat(z, mu[2])*vhat(z, mu[2]))*b[2] + 2*b[1])

It should be noted that in the FWM case the gauge transform that yielded meromorphic functions also reduced to $\rho(z)b_2 + b_1/2$ as it does in this case. It should also be noted that that is half the derivative of $b_0 + b_1\rho(z)+b_2\rho(z)^2$ w.r.t $\rho$.

After the gauge transform the 5 parameters are reduced to 2 and the equations of motion become:

In [180]:
du_dv_hat_eqs_b = [
    duhat_1_phi.subs([uv_phase_term_uv.args]),
    duhat_2_phi.subs([uv_phase_term_uv.args]),
    dvhat_1_phi.subs([uv_phase_term_uv.args]),
    dvhat_2_phi.subs([uv_phase_term_uv.args])
]

for _ in du_dv_hat_eqs_b:
    _


Eq(Derivative(uhat(z, mu[1]), z), -((uhat(z, mu[1])*vhat(z, mu[1]) + uhat(z, mu[2])*vhat(z, mu[2]))*b[2] - b[1])*uhat(z, mu[1])/2 + vhat(z, mu[2]))

Eq(Derivative(uhat(z, mu[2]), z), -((uhat(z, mu[1])*vhat(z, mu[1]) + uhat(z, mu[2])*vhat(z, mu[2]))*b[2] - b[1])*uhat(z, mu[2])/2 + vhat(z, mu[1]))

Eq(Derivative(vhat(z, mu[1]), z), ((uhat(z, mu[1])*vhat(z, mu[1]) + uhat(z, mu[2])*vhat(z, mu[2]))*b[2] - b[1])*vhat(z, mu[1])/2 - uhat(z, mu[2]))

Eq(Derivative(vhat(z, mu[2]), z), ((uhat(z, mu[1])*vhat(z, mu[1]) + uhat(z, mu[2])*vhat(z, mu[2]))*b[2] - b[1])*vhat(z, mu[2])/2 - uhat(z, mu[1]))

### The connection between $\varphi$ and $\phi$

In the unified theory we in general write a gauge term using $\phi$ as follows:

In [181]:
duz_unified = Eq(
    Derivative(u(z, mu[j]), z)/u(z, mu[j]),
    (rhop(z) - rhop(mu[j]))/(2*rho(z) - 2*rho(mu[j])) + Derivative(phi(z, mu[j]), z)
)
dvz_unified = Eq(
    Derivative(v(z, mu[j]), z)/v(z, mu[j]),
    (rhop(z) + rhop(mu[j]))/(2*rho(z) - 2*rho(mu[j])) - Derivative(phi(z, mu[j]), z)
)


duz_unified
dvz_unified

Eq(Derivative(u(z, mu[j]), z)/u(z, mu[j]), (\rho'(z) - \rho'(mu[j]))/(2*rho(z) - 2*rho(mu[j])) + Derivative(phi(z, mu[j]), z))

Eq(Derivative(v(z, mu[j]), z)/v(z, mu[j]), (\rho'(z) + \rho'(mu[j]))/(2*rho(z) - 2*rho(mu[j])) - Derivative(phi(z, mu[j]), z))

Which for the coherent coupler is:

In [191]:
dphi_m_Q = Eq(
    Derivative(phi(z, mu[m]), z),
    (Sum(a[m, k], (k, 1, 2)) - Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)))*rho(z) 
    - a[m] - gamma[m]*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2))
    + Sum(a[j, j]*gamma[j], (j, 1, 2))/2 - Sum(a[m, k]*gamma[k], (k, 1, 2)) + Sum(a[j]/2, (j, 1, 2))
)

dphi_m_Q_Lam = Eq(Derivative(phi(z, mu[m]), z), rho(z)*Lamda[1, m] + Lamda[0, m])
dphi_m_Q_Lam_sum = Eq(
    Sum(Derivative(phi(z, mu[j]), z), (j,1,2)), 
    rho(z)*b[2]
)

Lam0_sum_b = Eq(
    Sum(Lamda[0, m], (m, 1, 2)),
    (Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j, k]*gamma[k], (j, 1, 2), (k, 1, 2)))
    .subs(j,1).doit().simplify().subs(ajk_syms).subs(gamma[2], - gamma[1])
)
Lam1_sum_b = Eq(Sum(Lamda[1, m], (m, 1, 2)), b[2])


dphi_m_Q
dphi_m_Q_Lam
Lam0_sum_b
Lam1_sum_b
dphi_m_Q_Lam_sum

Eq(Derivative(phi(z, mu[m]), z), (Sum(a[m, k], (k, 1, 2)) - Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)))*rho(z) - a[m] - gamma[m]*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)) + Sum(a[j, j]*gamma[j], (j, 1, 2))/2 - Sum(a[m, k]*gamma[k], (k, 1, 2)) + Sum(a[j]/2, (j, 1, 2)))

Eq(Derivative(phi(z, mu[m]), z), rho(z)*Lamda[1, m] + Lamda[0, m])

Eq(Sum(Lamda[0, m], (m, 1, 2)), 0)

Eq(Sum(Lamda[1, m], (m, 1, 2)), b[2])

Eq(Sum(Derivative(phi(z, mu[j]), z), (j, 1, 2)), rho(z)*b[2])

The new equations of motion could thus be written:

In [200]:
du_dv_hat_eqs_d_phi = [
    duhat_1_phi.subs([uv_phase_term_rho.args, (dphi_m_Q_Lam_sum.rhs, dphi_m_Q_Lam_sum.lhs)]),
    duhat_2_phi.subs([uv_phase_term_rho.args, (dphi_m_Q_Lam_sum.rhs, dphi_m_Q_Lam_sum.lhs)]),
    dvhat_1_phi.subs([uv_phase_term_rho.args, (dphi_m_Q_Lam_sum.rhs, dphi_m_Q_Lam_sum.lhs)]),
    dvhat_2_phi.subs([uv_phase_term_rho.args, (dphi_m_Q_Lam_sum.rhs, dphi_m_Q_Lam_sum.lhs)])
]
du_dv_hat_eqs_d_phi = [
    Eq(_.lhs, _.rhs.collect(_.rhs.args[1], factor)) for _ in du_dv_hat_eqs_d_phi
]

for _ in du_dv_hat_eqs_d_phi:
    _


Eq(Derivative(uhat(z, mu[1]), z), (b[1] + 2*Sum(Derivative(phi(z, mu[j]), z), (j, 1, 2)))*uhat(z, mu[1])/2 + vhat(z, mu[2]))

Eq(Derivative(uhat(z, mu[2]), z), (b[1] + 2*Sum(Derivative(phi(z, mu[j]), z), (j, 1, 2)))*uhat(z, mu[2])/2 + vhat(z, mu[1]))

Eq(Derivative(vhat(z, mu[1]), z), (-b[1] - 2*Sum(Derivative(phi(z, mu[j]), z), (j, 1, 2)))*vhat(z, mu[1])/2 - uhat(z, mu[2]))

Eq(Derivative(vhat(z, mu[2]), z), (-b[1] - 2*Sum(Derivative(phi(z, mu[j]), z), (j, 1, 2)))*vhat(z, mu[2])/2 - uhat(z, mu[1]))

## Quartic to cubic

In [203]:
for _ in uv_hat_gauge_bs:
    _

Eq(uhat(z, mu[1]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - 2*z0 + mu[1], g2, g3)*exp(z*beta[0] + z*r[0, 1] + epsilon[1])*upsilon/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(-z0 + mu[1], g2, g3)*sigma(z - z0 - z1, g2, g3)*d[4]**(1/4)))

Eq(vhat(z, mu[1]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - mu[1], g2, g3)*exp(-z*beta[0] - z*r[0, 1] - epsilon[1])/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(-z0 + mu[1], g2, g3)*sigma(z - z0 + z1, g2, g3)*d[4]**(1/4)*upsilon))

Eq(uhat(z, mu[2]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - 2*z0 + mu[2], g2, g3)*exp(-z*beta[0] + z*r[0, 2] + epsilon[2])*upsilon/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(-z0 + mu[2], g2, g3)*sigma(z - z0 - z1, g2, g3)*d[4]**(1/4)))

Eq(vhat(z, mu[2]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - mu[2], g2, g3)*exp(z*beta[0] - z*r[0, 2] - epsilon[2])/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(-z0 + mu[2], g2, g3)*sigma(z - z0 + z1, g2, g3)*d[4]**(1/4)*upsilon))

In [258]:
u_v_hat_to_bar_eqs = [
    Eq(uhat(z, mu[1]), ubar(z, mu[1])/vbar(z, mu[3])),
    Eq(uhat(z, mu[2]), ubar(z, mu[2])/vbar(z, mu[3])),
    Eq(vhat(z, mu[1]), vbar(z, mu[1])/ubar(z, mu[3])),
    Eq(vhat(z, mu[2]), vbar(z, mu[2])/ubar(z, mu[3]))
]
u_v_hat_to_bar_subs = [_.args for _ in u_v_hat_to_bar_eqs]


for _ in u_v_hat_to_bar_eqs:
    _

Eq(uhat(z, mu[1]), ubar(z, mu[1])/vbar(z, mu[3]))

Eq(uhat(z, mu[2]), ubar(z, mu[2])/vbar(z, mu[3]))

Eq(vhat(z, mu[1]), vbar(z, mu[1])/ubar(z, mu[3]))

Eq(vhat(z, mu[2]), vbar(z, mu[2])/ubar(z, mu[3]))

In [301]:
_ubar1 = Eq(
    ubar(z, mu[1]), 
    sqrt(wpp(z1, g2, g3))*sigma(z1, g2, g3)
    *exp(epsilon[1])*upsilon/
    (sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(-z0 + mu[1], g2, g3)*d[4]**Rational(1,4))
    *sigma(z - 2*z0 + mu[1], g2, g3)*exp(z*beta[0] + z*r[0,1])/sigma(z - z0, g2, g3)
)
_ubar2 = Eq(
    ubar(z, mu[2]), 
    sqrt(wpp(z1, g2, g3))*sigma(z1, g2, g3)
    *exp(epsilon[2])*upsilon/
    (sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(-z0 + mu[2], g2, g3)*d[4]**Rational(1,4))
    *sigma(z - 2*z0 + mu[2], g2, g3)*exp(-z*beta[0] + z*r[0,2])/sigma(z - z0, g2, g3)
)
_ubar3 = Eq(
    ubar(z, mu[3]), 
    sigma(z - z0 + z1, g2, g3)/sigma(z - z0, g2, g3)
)
_vbar1 = Eq(
    vbar(z, mu[1]), 
    sqrt(wpp(z1, g2, g3))*sigma(z1, g2, g3)
    *exp(-epsilon[1])/
    (sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(-z0 + mu[1], g2, g3)*d[4]**Rational(1,4)*upsilon)
    *sigma(z - mu[1], g2, g3)*exp(-z*beta[0] - z*r[0,1])/sigma(z - z0, g2, g3)
)
_vbar2 = Eq(
    vbar(z, mu[2]), 
    sqrt(wpp(z1, g2, g3))*sigma(z1, g2, g3)*exp(-epsilon[2])/
    (sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(-z0 + mu[2], g2, g3)*d[4]**(1/4)*upsilon)
    *sigma(z - mu[2], g2, g3)*exp(z*beta[0] - z*r[0,2])/sigma(z - z0, g2, g3)
)

_vbar3 = Eq(
    vbar(z, mu[3]), 
    sigma(z - z0 - z1, g2, g3)/sigma(z - z0, g2, g3)
)

bar_sig_uv = [
    _ubar1,
    _ubar2,
    _ubar3,
    _vbar1,
    _vbar2,
    _vbar3
]
for _ in bar_sig_uv:
    _

print()
for _ in u_v_hat_to_bar_eqs:
    _.subs([_.args for _ in bar_sig_uv])

Eq(ubar(z, mu[1]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - 2*z0 + mu[1], g2, g3)*exp(z*beta[0] + z*r[0, 1])*exp(epsilon[1])*upsilon/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[1], g2, g3)*d[4]**(1/4)))

Eq(ubar(z, mu[2]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - 2*z0 + mu[2], g2, g3)*exp(-z*beta[0] + z*r[0, 2])*exp(epsilon[2])*upsilon/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[2], g2, g3)*d[4]**(1/4)))

Eq(ubar(z, mu[3]), sigma(z - z0 + z1, g2, g3)/sigma(z - z0, g2, g3))

Eq(vbar(z, mu[1]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - mu[1], g2, g3)*exp(-z*beta[0] - z*r[0, 1])*exp(-epsilon[1])/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[1], g2, g3)*d[4]**(1/4)*upsilon))

Eq(vbar(z, mu[2]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - mu[2], g2, g3)*exp(z*beta[0] - z*r[0, 2])*exp(-epsilon[2])/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[2], g2, g3)*d[4]**0.25*upsilon))

Eq(vbar(z, mu[3]), sigma(z - z0 - z1, g2, g3)/sigma(z - z0, g2, g3))

Eq(uhat(z, mu[1]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - 2*z0 + mu[1], g2, g3)*exp(z*beta[0] + z*r[0, 1])*exp(epsilon[1])*upsilon/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(-z0 + mu[1], g2, g3)*sigma(z - z0 - z1, g2, g3)*d[4]**(1/4)))

Eq(uhat(z, mu[2]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - 2*z0 + mu[2], g2, g3)*exp(-z*beta[0] + z*r[0, 2])*exp(epsilon[2])*upsilon/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(-z0 + mu[2], g2, g3)*sigma(z - z0 - z1, g2, g3)*d[4]**(1/4)))

Eq(vhat(z, mu[1]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - mu[1], g2, g3)*exp(-z*beta[0] - z*r[0, 1])*exp(-epsilon[1])/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(-z0 + mu[1], g2, g3)*sigma(z - z0 + z1, g2, g3)*d[4]**(1/4)*upsilon))

Eq(vhat(z, mu[2]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - mu[2], g2, g3)*exp(z*beta[0] - z*r[0, 2])*exp(-epsilon[2])/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(-z0 + mu[2], g2, g3)*sigma(z - z0 + z1, g2, g3)*d[4]**0.25*upsilon))

In [302]:
_c1 = Eq(
    uv_hat_gauge_bs[0].lhs/u_v_hat_to_bar_eqs[0].lhs.subs([_.args for _ in bar_sig_uv]), 
    (uv_hat_gauge_bs[0].rhs/u_v_hat_to_bar_eqs[0].rhs.subs([_.args for _ in bar_sig_uv]))
    .simplify()
)
_c3 = Eq(
    uv_hat_gauge_bs[1].lhs/u_v_hat_to_bar_eqs[2].lhs.subs([_.args for _ in bar_sig_uv]), 
    (uv_hat_gauge_bs[1].rhs/u_v_hat_to_bar_eqs[2].rhs.subs([_.args for _ in bar_sig_uv]))
    .simplify()
)

_c2 = Eq(
    uv_hat_gauge_bs[2].lhs/u_v_hat_to_bar_eqs[1].lhs.subs([_.args for _ in bar_sig_uv]), 
    (uv_hat_gauge_bs[2].rhs/u_v_hat_to_bar_eqs[1].rhs.subs([_.args for _ in bar_sig_uv]))
    .simplify()
)
_c4 = Eq(
    uv_hat_gauge_bs[3].lhs/u_v_hat_to_bar_eqs[3].lhs.subs([_.args for _ in bar_sig_uv]), 
    (uv_hat_gauge_bs[3].rhs/u_v_hat_to_bar_eqs[3].rhs.subs([_.args for _ in bar_sig_uv]))
    .simplify()
)
assert _c1
assert _c2
assert _c3
assert _c4

In [303]:
uv_hat_const = Eq(uhat(z, mu[1])*vhat(z, mu[1]) - uhat(z, mu[2])*vhat(z, mu[2]), gamma[1] - gamma[2])
uv_hat_bar_const = Eq(uv_hat_const.lhs.subs(u_v_hat_to_bar_subs), uv_hat_const.rhs)

uv_hat_const
uv_hat_bar_const

Eq(uhat(z, mu[1])*vhat(z, mu[1]) - uhat(z, mu[2])*vhat(z, mu[2]), gamma[1] - gamma[2])

Eq(ubar(z, mu[1])*vbar(z, mu[1])/(ubar(z, mu[3])*vbar(z, mu[3])) - ubar(z, mu[2])*vbar(z, mu[2])/(ubar(z, mu[3])*vbar(z, mu[3])), gamma[1] - gamma[2])

In [304]:
for _ in du_dv_hat_eqs_b:
    _.subs(u_v_hat_to_bar_subs).doit()


Eq(-ubar(z, mu[1])*Derivative(vbar(z, mu[3]), z)/vbar(z, mu[3])**2 + Derivative(ubar(z, mu[1]), z)/vbar(z, mu[3]), -((ubar(z, mu[1])*vbar(z, mu[1])/(ubar(z, mu[3])*vbar(z, mu[3])) + ubar(z, mu[2])*vbar(z, mu[2])/(ubar(z, mu[3])*vbar(z, mu[3])))*b[2] - b[1])*ubar(z, mu[1])/(2*vbar(z, mu[3])) + vbar(z, mu[2])/ubar(z, mu[3]))

Eq(-ubar(z, mu[2])*Derivative(vbar(z, mu[3]), z)/vbar(z, mu[3])**2 + Derivative(ubar(z, mu[2]), z)/vbar(z, mu[3]), -((ubar(z, mu[1])*vbar(z, mu[1])/(ubar(z, mu[3])*vbar(z, mu[3])) + ubar(z, mu[2])*vbar(z, mu[2])/(ubar(z, mu[3])*vbar(z, mu[3])))*b[2] - b[1])*ubar(z, mu[2])/(2*vbar(z, mu[3])) + vbar(z, mu[1])/ubar(z, mu[3]))

Eq(Derivative(vbar(z, mu[1]), z)/ubar(z, mu[3]) - vbar(z, mu[1])*Derivative(ubar(z, mu[3]), z)/ubar(z, mu[3])**2, ((ubar(z, mu[1])*vbar(z, mu[1])/(ubar(z, mu[3])*vbar(z, mu[3])) + ubar(z, mu[2])*vbar(z, mu[2])/(ubar(z, mu[3])*vbar(z, mu[3])))*b[2] - b[1])*vbar(z, mu[1])/(2*ubar(z, mu[3])) - ubar(z, mu[2])/vbar(z, mu[3]))

Eq(Derivative(vbar(z, mu[2]), z)/ubar(z, mu[3]) - vbar(z, mu[2])*Derivative(ubar(z, mu[3]), z)/ubar(z, mu[3])**2, ((ubar(z, mu[1])*vbar(z, mu[1])/(ubar(z, mu[3])*vbar(z, mu[3])) + ubar(z, mu[2])*vbar(z, mu[2])/(ubar(z, mu[3])*vbar(z, mu[3])))*b[2] - b[1])*vbar(z, mu[2])/(2*ubar(z, mu[3])) - ubar(z, mu[1])/vbar(z, mu[3]))

In [227]:
a_0_eq_bar = a_0_eq.doit().subs([
    uU_sub.subs(j,1).args, 
    uU_sub.subs(j,2).args, 
    vV_sub.subs(j,1).args, 
    vV_sub.subs(j,2).args,
    (varphi(z,2), -varphi(z,1))
]).subs(u_v_hat_to_bar_subs)

uvbar3 = ubar(z, mu[3])*vbar(z, mu[3])
a_0_eq_bar = Eq(
    sum((_*uvbar3**2).simplify() for _ in a_0_eq_bar.lhs.args),
    sum((_*uvbar3**2).simplify() for _ in a_0_eq_bar.rhs.args)
)
a_0_eq_bar

Eq((ubar(z, mu[1])*vbar(z, mu[1])*a[1, 1] + ubar(z, mu[2])*vbar(z, mu[2])*a[2, 1])*ubar(z, mu[1])*vbar(z, mu[1])/2 + (ubar(z, mu[1])*vbar(z, mu[1])*a[1, 2] + ubar(z, mu[2])*vbar(z, mu[2])*a[2, 2])*ubar(z, mu[2])*vbar(z, mu[2])/2 + ubar(z, mu[1])*ubar(z, mu[3])*vbar(z, mu[1])*vbar(z, mu[3])*a[1] + ubar(z, mu[2])*ubar(z, mu[3])*vbar(z, mu[2])*vbar(z, mu[3])*a[2] + ubar(z, mu[3])**2*vbar(z, mu[3])**2*a[0], ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])**2 + vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])**2)

In [231]:
diff(a_0_eq_bar.rhs - a_0_eq_bar.lhs, vbar(z, mu[3])).collect([vbar(z, mu[1])*vbar(z, mu[2])], factor)

-(ubar(z, mu[1])*vbar(z, mu[1])*a[1] + ubar(z, mu[2])*vbar(z, mu[2])*a[2] + 2*ubar(z, mu[3])*vbar(z, mu[3])*a[0])*ubar(z, mu[3]) + 2*vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])

In [234]:

_eq1 = du_dv_hat_eqs_b[0].subs(u_v_hat_to_bar_subs).doit()
_eq2 = du_dv_hat_eqs_b[1].subs(u_v_hat_to_bar_subs).doit()

_eq1
_eq2

Eq(-ubar(z, mu[1])*Derivative(vbar(z, mu[3]), z)/vbar(z, mu[3])**2 + Derivative(ubar(z, mu[1]), z)/vbar(z, mu[3]), -((ubar(z, mu[1])*vbar(z, mu[1])/(ubar(z, mu[3])*vbar(z, mu[3])) + ubar(z, mu[2])*vbar(z, mu[2])/(ubar(z, mu[3])*vbar(z, mu[3])))*b[2] - b[1])*ubar(z, mu[1])/(2*vbar(z, mu[3])) + vbar(z, mu[2])/ubar(z, mu[3]))

Eq(-ubar(z, mu[2])*Derivative(vbar(z, mu[3]), z)/vbar(z, mu[3])**2 + Derivative(ubar(z, mu[2]), z)/vbar(z, mu[3]), -((ubar(z, mu[1])*vbar(z, mu[1])/(ubar(z, mu[3])*vbar(z, mu[3])) + ubar(z, mu[2])*vbar(z, mu[2])/(ubar(z, mu[3])*vbar(z, mu[3])))*b[2] - b[1])*ubar(z, mu[2])/(2*vbar(z, mu[3])) + vbar(z, mu[1])/ubar(z, mu[3]))

In [239]:
Eq(
    _eq1.lhs*ubar(z,mu[3])*vbar(z, mu[1])*vbar(z, mu[3])*2,
    _eq1.rhs*ubar(z,mu[3])*vbar(z, mu[1])*vbar(z, mu[3])*2
).expand()

Eq(-2*ubar(z, mu[1])*ubar(z, mu[3])*vbar(z, mu[1])*Derivative(vbar(z, mu[3]), z)/vbar(z, mu[3]) + 2*ubar(z, mu[3])*vbar(z, mu[1])*Derivative(ubar(z, mu[1]), z), -ubar(z, mu[1])**2*vbar(z, mu[1])**2*b[2]/vbar(z, mu[3]) - ubar(z, mu[1])*ubar(z, mu[2])*vbar(z, mu[1])*vbar(z, mu[2])*b[2]/vbar(z, mu[3]) + ubar(z, mu[1])*ubar(z, mu[3])*vbar(z, mu[1])*b[1] + 2*vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3]))